In [1]:
# mtx2adata.ipynb - convert sparse matrix file into adata file.

In [2]:
import os
import pandas as pd
from sccnasim.xlib.xdata import load_10x_data

In [3]:
data_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/rdr_starsolo"
gene_anno_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/gene_anno/refdata-cellranger-GRCh38-3.0.0.valid_name.uniq_name.chrom1-22XY.5column.genes.tsv"

out_dir = data_dir
out_adata_fn = os.path.join(out_dir, "HCC3N_600spot.spotxgene.starsolo.h5ad")

## Construct adata

In [4]:
adata = load_10x_data(data_dir)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 33538
    obs: 'cell'
    var: 'feature_id', 'feature_name'

In [5]:
adata.obs['cell_type'] = 'normal'
adata.obs

,cell,cell_type
0,AAACAAGTATCTCCCA-1,normal
1,AAACATTTCCCGGATT-1,normal
2,AAACCGTTCGTCCAGG-1,normal
3,AAACCTAAGCAGCCGG-1,normal
4,AAACGGGCGTACGGGT-1,normal
...,...,...
595,TTGTGCAGCCACGTCA-1,normal
596,TTGTGGTGGTACTAAG-1,normal
597,TTGTGTATGCCACCAA-1,normal
598,TTGTGTTTCCCGAAAG-1,normal


In [6]:
adata.var.columns = ['gene_id', 'gene']
adata.var

,gene_id,gene
0,ENSG00000243485,MIR1302-2HG
1,ENSG00000237613,FAM138A
2,ENSG00000186092,OR4F5
3,ENSG00000238009,AL627309.1
4,ENSG00000239945,AL627309.3
...,...,...
33533,ENSG00000277856,AC233755.2
33534,ENSG00000275063,AC233755.1
33535,ENSG00000271254,AC240274.1
33536,ENSG00000277475,AC213203.1


In [7]:
gene_anno = pd.read_csv(gene_anno_fn, sep = '\t', header = None, dtype = {0:str})
gene_anno.columns = ['chrom', 'start', 'end', 'gene', 'strand']
gene_anno

,chrom,start,end,gene,strand
0,1,29554,31109,MIR1302-2HG,+
1,1,34554,36081,FAM138A,-
2,1,65419,71585,OR4F5,+
3,1,89295,133723,AL627309.1,-
4,1,89551,91105,AL627309.3,-
...,...,...,...,...,...
33467,Y,25063083,25099892,TTTY4C,-
33468,Y,25183643,25184773,TTTY17C,-
33469,Y,25378300,25394719,LINC00266-4P,-
33470,Y,25622162,25624902,CDY1,+


In [8]:
adata.var = adata.var.merge(gene_anno, on = 'gene', how = 'left')
adata.var

,gene_id,gene,chrom,start,end,strand
0,ENSG00000243485,MIR1302-2HG,1,29554.0,31109.0,+
1,ENSG00000237613,FAM138A,1,34554.0,36081.0,-
2,ENSG00000186092,OR4F5,1,65419.0,71585.0,+
3,ENSG00000238009,AL627309.1,1,89295.0,133723.0,-
4,ENSG00000239945,AL627309.3,1,89551.0,91105.0,-
...,...,...,...,...,...,...
33533,ENSG00000277856,AC233755.2,NaN,NaN,NaN,NaN
33534,ENSG00000275063,AC233755.1,NaN,NaN,NaN,NaN
33535,ENSG00000271254,AC240274.1,NaN,NaN,NaN,NaN
33536,ENSG00000277475,AC213203.1,NaN,NaN,NaN,NaN


## Save adata

In [9]:
adata.X = adata.X.toarray()

In [10]:
adata

AnnData object with n_obs × n_vars = 600 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [11]:
adata.write(out_adata_fn, compression = 'gzip')

In [12]:
adata.obs.to_csv(
    os.path.join(out_dir, 'spot_anno.tsv'),
    header = False,
    index = False,
    sep = '\t'
)